# Transformer 网络应用：命名实体识别

欢迎来到第 4 周的第二个不计分实验室。在此 Notebook 中，你将探索在之前的作业中构建的 Transformer 架构的一个应用程序。

**在此作业之后，你将能够**：

* 使用 HuggingFace 库中的 tokenizer 和预训练模型。
* 微调预训练 Transformer 模型以进行命名实体识别

## 目录

- [依赖库](#0)
- [1 - 处理简历的命名实体识别](#1)
    - [1.1 - 数据清理](#1-1)
    - [1.2 - 填充和生成标签](#1-2)
    - [1.3 - 使用 🤗 库进行分词并对齐标签](#1-3)
        - [练习 1 - tokenize_and_align_labels](#ex-1)
    - [1.4 - 优化](#1-4)

<a name='0'></a>
## 依赖库

运行以下单元格来加载你需要的包。

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import json
import random
import logging
import re

tf.get_logger().setLevel('ERROR')

<a name='1'></a>
## 1 - 处理简历的命名实体识别

当面对大量非结构化文本数据时，命名实体识别 (NER) 可以帮助你检测数据集中的重要信息并对其进行分类。例如，在运行示例“Jane vists Africa in September”中，NER 将帮助你将“Jane”、“Africa”和“September”检测为命名实体，并将它们分类为人物、位置和时间。 

* 你将使用在上一个作业中构建的 Transformer 模型的变体来处理大型简历数据集。
* 你将找到并分类相关信息，例如申请人工作过的公司、技能、学位类型等。 

<a name='1-1'></a>
### 1.1 - 数据集清理

在此作业中，你将在简历数据集上优化 Transformer 模型。查看你将使用的数据的结构。

In [ ]:
df_data = pd.read_json("ner.json", lines=True)
df_data = df_data.drop(['extras'], axis=1)
df_data['content'] = df_data['content'].str.replace("\n", " ")

In [ ]:
df_data.head()

In [ ]:
df_data.iloc[0]['annotation']

In [ ]:
def mergeIntervals(intervals):
    sorted_by_lower_bound = sorted(intervals, key=lambda tup: tup[0])
    merged = []

    for higher in sorted_by_lower_bound:
        if not merged:
            merged.append(higher)
        else:
            lower = merged[-1]
            if higher[0] <= lower[1]:
                if lower[2] is higher[2]:
                    upper_bound = max(lower[1], higher[1])
                    merged[-1] = (lower[0], upper_bound, lower[2])
                else:
                    if lower[1] > higher[1]:
                        merged[-1] = lower
                    else:
                        merged[-1] = (lower[0], higher[1], higher[2])
            else:
                merged.append(higher)
    return merged

In [ ]:
def get_entities(df):
    
    entities = []
    
    for i in range(len(df)):
        entity = []
    
        for annot in df['annotation'][i]:
            try:
                ent = annot['label'][0]
                start = annot['points'][0]['start']
                end = annot['points'][0]['end'] + 1
                entity.append((start, end, ent))
            except:
                pass
    
        entity = mergeIntervals(entity)
        entities.append(entity)
    
    return entities

In [ ]:
df_data['entities'] = get_entities(df_data)
df_data.head()

In [ ]:
def convert_dataturks_to_spacy(dataturks_JSON_FilePath):
    try:
        training_data = []
        lines=[]
        with open(dataturks_JSON_FilePath, 'r') as f:
            lines = f.readlines()

        for line in lines:
            data = json.loads(line)
            text = data['content'].replace("\n", " ")
            entities = []
            data_annotations = data['annotation']
            if data_annotations is not None:
                for annotation in data_annotations:
                    #文本注释中只有一个点。
                    point = annotation['points'][0]
                    labels = annotation['label']
                    # 处理标签列表或单个标签。
                    if not isinstance(labels, list):
                        labels = [labels]

                    for label in labels:
                        point_start = point['start']
                        point_end = point['end']
                        point_text = point['text']
                        
                        lstrip_diff = len(point_text) - len(point_text.lstrip())
                        rstrip_diff = len(point_text) - len(point_text.rstrip())
                        if lstrip_diff != 0:
                            point_start = point_start + lstrip_diff
                        if rstrip_diff != 0:
                            point_end = point_end - rstrip_diff
                        entities.append((point_start, point_end + 1 , label))
            training_data.append((text, {"entities" : entities}))
        return training_data
    except Exception as e:
        logging.exception("Unable to process " + dataturks_JSON_FilePath + "\n" + "error = " + str(e))
        return None

def trim_entity_spans(data: list) -> list:
    """从实体跨度中删除前导和尾随空格。

    参数：
        data (list)：需要清理的数据，spaCy JSON格式。

    返回：
        list: 清理后的数据。
    """
    invalid_span_tokens = re.compile(r'\s')

    cleaned_data = []
    for text, annotations in data:
        entities = annotations['entities']
        valid_entities = []
        for start, end, label in entities:
            valid_start = start
            valid_end = end
            while valid_start < len(text) and invalid_span_tokens.match(
                    text[valid_start]):
                valid_start += 1
            while valid_end > 1 and invalid_span_tokens.match(
                    text[valid_end - 1]):
                valid_end -= 1
            valid_entities.append([valid_start, valid_end, label])
        cleaned_data.append([text, {'entities': valid_entities}])
    return cleaned_data  

In [ ]:
data = trim_entity_spans(convert_dataturks_to_spacy("ner.json"))

In [ ]:
from tqdm.notebook import tqdm
def clean_dataset(data):
    cleanedDF = pd.DataFrame(columns=["setences_cleaned"])
    sum1 = 0
    for i in tqdm(range(len(data))):
        start = 0
        emptyList = ["Empty"] * len(data[i][0].split())
        numberOfWords = 0
        lenOfString = len(data[i][0])
        strData = data[i][0]
        strDictData = data[i][1]
        lastIndexOfSpace = strData.rfind(' ')
        for i in range(lenOfString):
            if (strData[i]==" " and strData[i+1]!=" "):
                for k,v in strDictData.items():
                    for j in range(len(v)):
                        entList = v[len(v)-j-1]
                        if (start>=int(entList[0]) and i<=int(entList[1])):
                            emptyList[numberOfWords] = entList[2]
                            break
                        else:
                            continue
                start = i + 1  
                numberOfWords += 1
            if (i == lastIndexOfSpace):
                for j in range(len(v)):
                        entList = v[len(v)-j-1]
                        if (lastIndexOfSpace>=int(entList[0]) and lenOfString<=int(entList[1])):
                            emptyList[numberOfWords] = entList[2]
                            numberOfWords += 1
        cleanedDF = cleanedDF.append(pd.Series([emptyList],  index=cleanedDF.columns ), ignore_index=True )
        sum1 = sum1 + numberOfWords
    return cleanedDF

In [ ]:
cleanedDF = clean_dataset(data)

查看已清理的数据集以及命名实体匹配的类别或“标签”。

In [ ]:
cleanedDF.head()

<a name='1-2'></a>
### 1.2 - 填充和生成标签

现在，生成去重后的标签列表，用于标注各类命名实体。

In [ ]:
unique_tags = set(cleanedDF['setences_cleaned'].explode().unique())#pd.unique(cleanedDF['setences_cleaned'])#set（cleanedDF 中的文档标签['setences_cleaned'].values.tolist() 文档中的标签）
tag2id = {tag: id for id, tag in enumerate(unique_tags)}
id2tag = {id: tag for tag, id in tag2id.items()}

In [ ]:
unique_tags

接下来，你将从清理后的数据集中创建一系列标签。通常，你的输入序列可能会超出网络可以处理的序列的最大长度，因此需要将其截断到所需的最大长度。当输入序列短于所需长度时，你需要使用 [Keras 填充 API](https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/sequence/pad_sequences) 在其末尾附加零。

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
MAX_LEN = 512
labels = cleanedDF['setences_cleaned'].values.tolist()

tags = pad_sequences([[tag2id.get(l) for l in lab] for lab in labels],
                     maxlen=MAX_LEN, value=tag2id["Empty"], padding="post",
                     dtype="long", truncating="post")

In [ ]:
tags

<a name='1-3'></a>
### 1.3 - 使用 🤗 库进行分词并对齐标签

将文本输入 Transformer 模型前，需要使用 [🤗 Transformer tokenizer](https://huggingface.co/transformers/main_classes/tokenizer.html) 进行分词。tokenizer 必须与所用的 Transformer 模型匹配。本练习使用 🤗 [DistilBERT fast tokenizer](https://huggingface.co/transformers/model_doc/distilbert.html)，将序列统一为 512 个 token，不足部分用 0 填充。这与创建标签时使用的最大长度一致。

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_virtual_device_configuration(gpu,[tf.config.experimental.VirtualDeviceConfiguration(memory_limit=4096)])

In [ ]:
from transformers import DistilBertTokenizerFast #，TFDistilBertModel
tokenizer = DistilBertTokenizerFast.from_pretrained('tokenizer/')

Transformer 模型通常配套使用能将单词拆分为子词的 tokenizer。例如，"Africa" 可能被拆为多个子词 token。这会导致数据集中按词标注的标签与分词结果不再一一对应；tokenizer 还可能加入特殊 token。因此，需要使用 `tokenize_and_align_labels()` 将原始标签与分词结果对齐。

<a name='ex-1'></a>
### 练习 1 - tokenize_and_align_labels

实现`tokenize_and_align_labels()`。 函数应执行以下操作：
* 使用 tokenizer 的 `truncation=True` 参数，截断超出模型最大长度的序列。
* 使用 tokenizer 的 `word_ids` 方法获取映射：普通 token 对应其原始单词的索引，特殊 token 对应 `None`。
* 将特殊 token（映射为 `None`）的标签设为 -100，使损失函数忽略这些位置。
* 将单词的原始标签赋给它的第一个子词 token，其余子词 token 的标签设为 -100。

In [ ]:
label_all_tokens = True
def tokenize_and_align_labels(tokenizer, examples, tags):
    tokenized_inputs = tokenizer(examples, truncation=True, is_split_into_words=False, padding='max_length', max_length=512)
    labels = []
    for i, label in enumerate(tags):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # 特殊标记的单词 ID 为 None。我们将标签设置为 -100，以便它们自动
            # 在损失函数中被忽略。
            if word_idx is None:
                label_ids.append(-100)
            # 我们为每个单词的第一个标记设置标签。
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # 对于单词中的其他标记，我们将标签设置为当前标签或-100，具体取决于
            # label_all_tokens 标志。
            else:
                label_ids.append(label[word_idx] if label_all_tokens else -100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

现在你已经分词了输入，你可以创建训练和测试数据集！

In [ ]:
test = tokenize_and_align_labels(tokenizer, df_data['content'].values.tolist(), tags)
train_dataset = tf.data.Dataset.from_tensor_slices((
    test['input_ids'],
    test['labels']
))

<a name='1-4'></a>
### 1.4 - 优化

太棒了！现在你终于可以将数据输入到预训练 🤗 模型中。你将优化 DistilBERT 模型，该模型与你用于预处理数据的 tokenizer 相匹配。尝试使用不同的超参数来改善你的结果！

In [ ]:
from transformers import TFDistilBertForTokenClassification

model = TFDistilBertForTokenClassification.from_pretrained('model/', num_labels=len(unique_tags))

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-5)
model.compile(optimizer=optimizer, loss=model.hf_compute_loss, metrics=['accuracy']) # 也可以使用任何Keras损失fn
model.fit(train_dataset.batch(4),
          epochs=10, 
          batch_size=4)

In [ ]:
text = "Manisha Bharti. 3.5 years of professional IT experience in Banking and Finance domain"
inputs = tokenizer(text, return_tensors="tf", truncation=True, is_split_into_words=False, padding="max_length", max_length=512 )
input_ids = inputs["input_ids"]
#inputs["labels"] = tf.reshape(tf.constant([1] * tf.size(input_ids).numpy()), (-1, tf.size(input_ids)))

In [ ]:
output = model(inputs).logits
prediction = np.argmax(output, axis=2)
print( prediction)

In [ ]:
model(inputs)

In [ ]:
pred_labels = []

In [ ]:
%pip install seqeval==1.2.2

In [ ]:
true_labels = [[id2tag.get(true_index, "Empty") for true_index in test['labels'][i]] for i in range(len(test['labels']))]
np.array(true_labels).shape

In [ ]:
output = model.predict(train_dataset)

In [ ]:
predictions = np.argmax(output['logits'].reshape(220, -1, 12), axis=-1)

In [ ]:
predictions.shape

In [ ]:
from matplotlib import pyplot as plt 

p = plt.hist(np.array(true_labels).flatten())
plt.xticks(rotation='vertical')
plt.show()

In [ ]:
from collections import Counter
Counter(np.array(true_labels).flatten())

In [ ]:
pred_labels = [[id2tag.get(index, "Empty") for index in predictions[i]] for i in range(len(predictions))]
p = plt.hist(np.array(pred_labels).flatten())
plt.xticks(rotation='vertical')
plt.show()

In [ ]:
from seqeval.metrics import classification_report
print(classification_report(true_labels, pred_labels))

### 恭喜！

#### 这是你应该记住的

- 命名实体识别 (NER) 对命名实体进行检测和分类，可以帮助处理简历、客户评论、浏览历史记录等。 
- 在将输入输入到 Transformer 模型之前，你必须使用与预训练模型相对应的 tokenizer 来预处理文本数据。